# ML TEMPLATE — FORECASTING

---

**Dieses Template enthält alle wichtigen Schritte für ein Zeitreihen-Forecasting-Projekt.**
Der Ansatz: Von einfach zu komplex. Statistische Modelle, klassisches ML und Deep Learning systematisch vergleichen.

| Schritt | Inhalt |
|---------|--------|
| 1 | Imports & Konfiguration |
| 2 | Daten laden & bereinigen |
| 3 | Explorative Datenanalyse (EDA) |
| 4 | Feature Engineering & Feiertags-Kalender |
| 5 | STL Decomposition |
| 6 | Ausreisser & Anomalien |
| 7 | Train/Test Split & Datenformate |
| 8 | Baseline Modell (Benchmark) |
| 9 | Statistische Modelle |
| 10 | Klassische ML-Modelle |
| 11 | Neurale Modelle |
| 12 | Cross-Validation |
| 13 | Evaluation & Modellvergleich |
| 14 | Wochentags-Analyse & Fehleranalyse |
| 15 | SHAP Explainability |
| 16 | Hyperparameter-Tuning (bestes Modell) |
| 17 | Finales Training |
| 18 | Modell speichern & laden |
| 19 | Ergebnisse nach Snowflake schreiben |
| 20 | Zusammenfassung & nächste Schritte |

---

## 1. Imports & Konfiguration
- Alle benötigten Libraries importieren
- Random Seed setzen für Reproduzierbarkeit
- Warnungen konfigurieren
- Plotting-Style definieren
- Config laden (Tabellennamen, Horizont, Features aus `configs/config.yaml`)

---

## 2. Daten laden & bereinigen
- Zeitreihen-Daten aus Snowflake laden mit `load_timeseries()`
- Erste Übersicht: Shape, Zeitraum, Anzahl Serien
- Bereinigung:
  - Wochenenden ausschliessen (falls Geschäftstage)
  - Tage mit Wert = 0 prüfen (echte Null oder fehlend?)
  - Fehlende Tage auffüllen (Median, Forward Fill oder 0)
- Zeitreihe plotten für visuellen Überblick

---

## 3. Explorative Datenanalyse (EDA)
- Statistische Zusammenfassung (`.describe()`)
- Fehlende Werte und Lücken in der Zeitreihe prüfen
- Verteilung der Zielvariable (Histogramm, Boxplot)
- Saisonalität erkennen:
  - Wochentags-Muster (Boxplot pro Wochentag)
  - Monats-Durchschnitte (Barplot pro Monat)
  - Jahresvergleich (gleicher Monat, verschiedene Jahre)
- Korrelationsanalyse

---

## 4. Feature Engineering & Feiertags-Kalender
- Feiertags-Kalender konfigurieren (Land, Kanton/Region)
- Kalender-Features: Wochentag, Monat, Wochenende, Quartal
- Feiertags-Features: is_holiday, day_before_holiday, is_short_week
- Monatsrand-Features: is_first_bday_of_month, is_last_bday_of_month
- Saisonale Features: is_july (Sommerferien), Dezember-Effekte
- Lag-Features: y_lag_1, y_lag_5, y_lag_10, y_lag_20
- Rolling-Features: Rolling Mean und Std (5, 10, 20 Tage)
- Wichtig: Nur Features verwenden, die in der Zukunft bekannt sind (für exogene Features)

---

## 5. STL Decomposition
- Zeitreihe in Trend, Saisonalität und Residuum zerlegen
- Saisonale Periode bestimmen (7 für Woche, 5 für Geschäftswoche, 12 für Monate)
- Visuell prüfen: Ist der Trend stabil? Wie stark ist die Saisonalität?
- Gibt es strukturelle Brüche (COVID, Umzüge, Systemwechsel)?

---

## 6. Ausreisser & Anomalien
- Ausreisser identifizieren (IQR-Methode, Z-Score)
- Sonderereignisse dokumentieren (Feiertage, Betriebsferien, Sonderaktionen)
- Entscheidung: Ausreisser entfernen, ersetzen oder belassen?
- Auswirkung auf das Modell testen

---

## 7. Train/Test Split & Datenformate
- **Nie** zufällig splitten bei Zeitreihen!
- Die letzten N Perioden als Test-Set (z.B. 14, 30 oder 60 Tage)
- Datenformate vorbereiten:
  - **NeuralForecast:** DataFrame mit [unique_id, ds, y] + exogene Spalten (FUTR_EXOG)
  - **StatsForecast:** DataFrame mit [unique_id, ds, y]
  - **MLForecast/LightGBM:** DataFrame mit Features + Lag-Spalten
  - **SARIMAX:** Einzelne Zeitreihe (y) + exogene Matrix (X)

---

## 8. Baseline Modell (Benchmark)
- **Immer als erstes erstellen!** Jedes ML-Modell muss die Baseline schlagen.
- Optionen:
  - Naive: Letzter bekannter Wert
  - Seasonal Naive: Wert vom gleichen Wochentag letzte Woche (y_lag_7 / y_lag_5)
  - Vorjahres-Matching: ISO-Kalenderwoche + Wochentag vom Vorjahr
  - Historischer Durchschnitt (Mean/Median)
- Metriken berechnen: MAE, RMSE, MAPE
- Baseline-Metriken dokumentieren

---

## 9. Statistische Modelle
Klassische Zeitreihenmodelle als solide Grundlage. Oft unterschätzt, manchmal die beste Wahl.

- **SARIMAX:** Saisonaler ARIMA mit exogenen Variablen
  - Order (p,d,q) und Seasonal Order (P,D,Q,s) bestimmen
  - Grid Search oder Auto-ARIMA für Parameter
- **AutoARIMA:** Automatische Parametersuche (StatsForecast)
- **ETS (Exponential Smoothing):** Error, Trend, Seasonality Zerlegung
- **Theta:** Theta-Methode (gut für kurzfristige Prognosen)
- **CES (Complex Exponential Smoothing)**
- **OLS Regression:** Lineare Regression mit Zeitreihen-Features als einfache Baseline

Alle statistischen Modelle mit `statsmodels` oder `StatsForecast` (Nixtla) trainieren.

---

## 10. Klassische ML-Modelle
Gradient Boosting auf Feature-Tabellen. Oft überraschend stark bei Zeitreihen.

- **LightGBM:** Über `MLForecast` (Nixtla) mit rekursiver Vorhersage oder direkt mit Lag-Features
- **XGBoost:** Alternative zu LightGBM
- Wichtig: Lag-Features korrekt definieren (keine Datenlecks!)
- Feature Importance analysieren

---

## 11. Neurale Modelle
Deep Learning Modelle für Zeitreihen. Brauchen das NeuralForecast-Format [unique_id, ds, y] + FUTR_EXOG.

- **N-HiTS:** Hierarchische Interpolation, sehr gut für Long-Horizon Forecasting
- **N-BEATS:** Neural Basis Expansion (ohne exogene Features, rein univariat)
- **PatchTST:** Transformer-basiert, Patching für Effizienz (ohne exogene Features)
- **TFT (Temporal Fusion Transformer):** Transformer mit Attention, nutzt exogene Features
- **TSMixerx:** MLP-Mixer Architektur mit exogenen Features
- **TiDE:** Token-based Temporal Encoder/Decoder
- **NeuralProphet:** Prophet-Nachfolger auf PyTorch-Basis

Alle neuralen Modelle über `NeuralForecast` (Nixtla) trainieren.
Nicht alle Modelle unterstützen exogene Features. Dokumentieren welche mit/ohne FUTR_EXOG laufen.

---

## 12. Cross-Validation
- TimeSeriesSplit oder NeuralForecast `cross_validation()` mit n_windows
- Typisch 3–5 Windows
- Mehrere Metriken pro Window: MAE, RMSE, MAPE
- Varianz der Ergebnisse prüfen (hohe Varianz = instabiles Modell)
- Nur für die vielversprechendsten Modelle durchführen (spart Rechenzeit)

---

## 13. Evaluation & Modellvergleich
- Vergleichstabelle aller Modelle erstellen
  - Sortiert nach MAE (niedrigster = bester)
  - Spalten: Modell, Typ, MAE, RMSE, MAPE, R²
- Visueller Vergleich: Forecast vs. Aktual Plot (alle Modelle übereinander)
- Welches Modell performt wann am besten? (z.B. Montags, Feiertage, Monatsende)

---

## 14. Wochentags-Analyse & Fehleranalyse
- MAE pro Wochentag aufschlüsseln
- Fehler an Feiertagen vs. normalen Tagen
- Fehler am Monatsanfang/Monatsende
- Systematische Abweichungen identifizieren
- Rückschlüsse für Feature Engineering ziehen

---

## 15. SHAP Explainability
- SHAP Values für das beste Modell (falls ML-basiert)
- Feature-Beiträge pro Vorhersage verstehen
- Business-Interpretation: Machen die wichtigsten Features Sinn?
- Für neurale Modelle: Feature Importance über NeuralForecast Methoden

---

## 16. Hyperparameter-Tuning (bestes Modell)
- Nur für das vielversprechendste Modell aus dem Vergleich
- Methoden:
  - **SARIMAX:** Grid Search über (p,d,q) und (P,D,Q,S)
  - **LightGBM/XGBoost:** Optuna oder GridSearchCV
  - **Neurale Modelle:** Optuna über NeuralForecast-Parameter (Stacks, Pooling, Learning Rate, etc.)
- Suchraum dokumentieren
- Beste Parameter speichern
- Optional: Ensemble-Test (Kombination der besten Modelle)

---

## 17. Finales Training
- Bestes Modell mit optimierten Parametern
- Training auf dem gesamten Datensatz (Train + Test)
- Oder: Refit mit erweiterten Daten

---

## 18. Modell speichern & laden
- Modell speichern:
  - SARIMAX: `pickle` oder `joblib`
  - LightGBM: `joblib` oder `model.save_model()`
  - NeuralForecast: `nf.save(path=...)`
- Versionierung: v1, v2, ... mit Datum im Namen
- Laden und verifizieren: Predictions auf bekannten Daten vergleichen
- In Snowflake: Modell auf Stage oder in Model Registry speichern

---

## 19. Ergebnisse nach Snowflake schreiben
- Forecast-Output standardisieren (Timestamp, Modellname, Version)
- Forecast-Tabelle schreiben (append, nicht überschreiben)
- Metriken loggen
- Experiment dokumentieren mit `log_experiment()`

---

## 20. Zusammenfassung & nächste Schritte
- Ergebnisse dokumentieren:
  - Bestes Modell, Typ, Metriken
  - Alle getesteten Modelle mit Ergebnissen
  - Wichtigste Features
- Empfehlung: Weiterentwickeln / Anpassen / Verwerfen?
- Nächste Schritte:
  - Weitere Serien / Granularitäten testen
  - Feature Engineering erweitern
  - Validierung auf neuen Daten (Out-of-Time)
  - Snowflake ML FORECAST als SQL-Baseline (`sql/03_snowflake_forecast.sql`)
  - Operationalisierung (ML Jobs + Tasks für automatische Forecasts)
  - Zukunftsprognose generieren und exportieren

---

*Template Version 1.0 | BIDA ML Starter | Pistor BI & Data Analytics*